# Music tagging with frozen audio embeddings

An MLP predicts 50 MagnaTagATune tags from frozen audio embeddings. This is a curated version of Manuel Castillo's Lab 9 for Machine Learning for Music at Universitat Pompeu Fabra.

Install the repository dependencies before running. Set `MTAT_ROOT` to your local dataset directory. No dataset, model weights or cached embeddings are included.

Historical results and their source cells are in [results.md](../docs/results.md). The cells below have been edited and their old outputs cleared; they have not been retrained during curation.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "music_experiments").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the repository.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from music_experiments.cache import cache_digest, source_row_indices


## Imports

In [ ]:
import os
import json
import time
import math
import random
import platform
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import torchaudio
import matplotlib.pyplot as plt

from tqdm.auto import tqdm, trange
from torch.utils.data import Dataset, DataLoader

import psutil
from codecarbon import OfflineEmissionsTracker

from omar_rq import get_model

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import roc_auc_score, average_precision_score

from IPython.display import Audio, display


## Device and seed

In [ ]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("Device:", device)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)


## Configuration

The original experiment used 16 kHz, two-second crops and a cap of 8,000 / 2,000 / 2,000 clips. These settings are retained for traceability, not as recommended encoder settings. Confirm the sample rate required by the audio encoder before a new benchmark.

Cache keys include the ordered clip IDs, labels and extraction settings. Set `data_revision` to a new value if you replace the local audio files. Each execution gets a separate output directory; training does not automatically resume an earlier run.

In [ ]:
from datetime import datetime
CFG = {
    "seed": 42,
    "sample_rate": 16000,
    "clip_seconds": 2.0,
    "top_k_tags": 50,
    "max_train_items": 8000,
    "max_val_items": 2000,
    "max_test_items": 2000,
    "repr_model_id": "mtg-upf/clap_omarrq_mp_small_music",
    "emb_layers": [6],
    "batch_size": 32,
    "mlp_hidden": 256,
    "mlp_dropout": 0.2,
    "epochs": 100,
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "early_stop_patience": 3,
    "eval_every": 3,
    "num_workers": 0,
    "pin_memory": device.type == "cuda",
    "data_revision": "local-v1",
    "track_emissions": False,
}
seed_everything(CFG["seed"])
cache_dir = REPO_ROOT / "artifacts" / "tagging" / "cache"
run_dir = REPO_ROOT / "artifacts" / "tagging" / "runs" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
cache_dir.mkdir(parents=True, exist_ok=True)
run_dir.mkdir(parents=True, exist_ok=True)
(run_dir / "config.json").write_text(json.dumps(CFG, indent=2))
# Preserve plain-text logging without ANSI styling.
class color:
    BOLD = END = YELLOW = GREEN = ""
print(CFG)


## Resource sampling

RAM and CPU samples are system-wide, not isolated model measurements. Sampling occurs at selected steps rather than at fixed intervals. Optional CodeCarbon estimates use `CODECARBON_COUNTRY_ISO_CODE` (default: ESP). This is not a hardware power benchmark.

In [ ]:
@dataclass
class ResourceLog:
    timestamps_s: list
    cpu_percent: list
    ram_used_gb: list
    ram_percent: list
    cuda_mem_alloc_gb: list
    cuda_mem_reserved_gb: list

def start_resource_logging(sample_every_s=2.0):
    log = ResourceLog([], [], [], [], [], [])
    start_t = time.time()

    def sample():
        t = time.time() - start_t
        cpu = psutil.cpu_percent(interval=None)
        mem = psutil.virtual_memory()
        ram_used = (mem.total - mem.available) / (1024**3)
        ram_pct = mem.percent

        cuda_alloc = 0.0
        cuda_res = 0.0
        if torch.cuda.is_available():
            cuda_alloc = torch.cuda.memory_allocated() / (1024**3)
            cuda_res = torch.cuda.memory_reserved() / (1024**3)

        log.timestamps_s.append(float(t))
        log.cpu_percent.append(float(cpu))
        log.ram_used_gb.append(float(ram_used))
        log.ram_percent.append(float(ram_pct))
        log.cuda_mem_alloc_gb.append(float(cuda_alloc))
        log.cuda_mem_reserved_gb.append(float(cuda_res))

    return log, sample, sample_every_s


resource_log, resource_sample, resource_period_s = start_resource_logging()
emissions_tracker = None
if CFG["track_emissions"]:
    emissions_tracker = OfflineEmissionsTracker(
        project_name="music_tagging", output_dir=str(run_dir),
        country_iso_code=os.getenv("CODECARBON_COUNTRY_ISO_CODE", "ESP"),
        measure_power_secs=15, log_level="error",
    )
    emissions_tracker.start()
resource_sample()


## Local dataset and split

Expected layout: `annotations_final.csv`, `clip_info_final.csv`, and `audio/` (or `mp3/`) with the paths listed in the metadata.

The original protocol shuffles clips with seed 42, splits 80/10/10 and applies the item caps. It does not group by song or artist. The top tags are selected using all available clips before splitting. These are limitations of the historical experiment; its numbers should not be compared directly with a benchmark using a different split.

In [ ]:
import csv
from dataclasses import dataclass

TOP_K_TAGS = int(CFG.get("top_k_tags", 50))  # from CFG

def _has_mtat_layout(root: Path) -> bool:
    root = Path(root)
    return (
        (root / "annotations_final.csv").is_file()
        and (root / "clip_info_final.csv").is_file()
        and ((root / "audio").is_dir() or (root / "mp3").is_dir())
    )

def _read_csv_dicts(path: Path):
    path = Path(path)
    with open(path, "r", encoding="utf-8", newline="") as f:
        first = f.readline()
    delim = "\t" if ("\t" in first and "," not in first) else ","
    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f, delimiter=delim)
        return list(reader)

def _find_mtat_root():
    value = os.environ.get("MTAT_ROOT", "").strip()
    if not value:
        raise FileNotFoundError("Set MTAT_ROOT to the directory containing annotations, metadata and audio/.")
    root = Path(value).expanduser().resolve()
    if not _has_mtat_layout(root):
        raise FileNotFoundError(f"Incomplete MTAT layout: {root}")
    return root

def _resolve_audio_root(base: Path) -> Path:
    base = Path(base)
    for c in [base / "audio", base / "mp3", base.parent / "audio", base.parent / "mp3"]:
        if c.exists() and c.is_dir():
            return c
    raise FileNotFoundError("audio folder not found near MTAT root")

def _normalize_mp3_rel(mp3_path: str) -> str:
    s = str(mp3_path).strip()
    if s.startswith("audio/"):
        s = s[len("audio/"):]
    if s.startswith("./"):
        s = s[2:]
    return s

def _to01(x):
    try:
        return int(float(str(x).strip()))
    except Exception:
        return 0

def _limit_list(xs, k):
    k = int(k) if k is not None else 0
    if k <= 0:
        return xs
    return xs[: min(len(xs), k)]

base = _find_mtat_root()
ann_path  = base / "annotations_final.csv"
info_path = base / "clip_info_final.csv"
audio_root = _resolve_audio_root(base)

print(color.BOLD + "MTAT root:" + color.END, base)
print("annotations:", ann_path)
print("clip_info:", info_path)
print("audio_root:", audio_root)

ann_rows  = _read_csv_dicts(ann_path)
info_rows = _read_csv_dicts(info_path)

if len(ann_rows) == 0 or len(info_rows) == 0:
    raise ValueError("annotations_final.csv or clip_info_final.csv is empty/unreadable.")

if "clip_id" not in ann_rows[0] or "clip_id" not in info_rows[0]:
    raise ValueError("Expected column 'clip_id' in both annotations_final.csv and clip_info_final.csv")

if "mp3_path" not in info_rows[0]:
    print("clip_info columns:", list(info_rows[0].keys()))
    raise ValueError("Expected column 'mp3_path' in clip_info_final.csv")

tag_cols_all = [c for c in ann_rows[0].keys() if c != "clip_id"]

clip_to_mp3 = {}
for r in info_rows:
    cid = str(r["clip_id"]).strip()
    clip_to_mp3[cid] = _normalize_mp3_rel(r["mp3_path"])

clip_to_y = {}
for r in ann_rows:
    cid = str(r["clip_id"]).strip()
    clip_to_y[cid] = [_to01(r.get(c, 0)) for c in tag_cols_all]

valid_cids = []
for cid, mp3_rel in clip_to_mp3.items():
    if cid not in clip_to_y:
        continue
    fp = audio_root / mp3_rel
    if fp.is_file():
        valid_cids.append(cid)

if len(valid_cids) == 0:
    raise ValueError("No valid clips found (mp3 files missing). Check audio_root and mp3_path normalization.")

Y_full = np.asarray([clip_to_y[cid] for cid in valid_cids], dtype=np.int32)
freq = Y_full.sum(axis=0)
top_idx = np.argsort(freq)[::-1][:TOP_K_TAGS]
tag_cols_top = [tag_cols_all[i] for i in top_idx]

@dataclass
class LocalMTATTrack:
    path: Path
    tags: list

    @property
    def audio(self):
        wav, sr = torchaudio.load(str(self.path))
        if wav.ndim == 2 and wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        wav = wav.squeeze(0).cpu().numpy()
        return wav, sr

tracks = {}
for cid in valid_cids:
    mp3_rel = clip_to_mp3[cid]
    fp = audio_root / mp3_rel
    y = np.asarray(clip_to_y[cid], dtype=np.int32)[top_idx]
    tags = [t for t, v in zip(tag_cols_top, y) if v == 1]
    tracks[cid] = LocalMTATTrack(path=fp, tags=tags)

track_ids = list(tracks.keys())
print(color.BOLD + "Total valid tracks:" + color.END, len(track_ids))
print("Top tags sample:", tag_cols_top[:15])

seed_everything(CFG["seed"])
random.shuffle(track_ids)

n = len(track_ids)
train_ids = track_ids[: int(0.8 * n)]
val_ids   = track_ids[int(0.8 * n): int(0.9 * n)]
test_ids  = track_ids[int(0.9 * n):]

train_ids = _limit_list(train_ids, CFG.get("max_train_items", 0))
val_ids   = _limit_list(val_ids,   CFG.get("max_val_items", 0))
test_ids  = _limit_list(test_ids,  CFG.get("max_test_items", 0))

print(color.BOLD + "Final split sizes (after limits)" + color.END)
print("train:", len(train_ids), "| val:", len(val_ids), "| test:", len(test_ids))

len(train_ids), len(val_ids), len(test_ids)
(run_dir / "split_ids.json").write_text(json.dumps({"train": train_ids, "validation": val_ids, "test": test_ids}, indent=2))


## Audio loading

Decode with torchaudio, then try FFmpeg if needed. An unreadable file raises an error instead of being replaced with silence. Inspect or replace that file before restarting. Each clip gets a deterministic crop based on its ID and the configured seed.

In [ ]:
SR = int(CFG.get("sample_rate", 16000))
CLIP_S = float(CFG.get("clip_seconds", 2.0))
INSPECT_S = float(min(3.0, CLIP_S))

def load_track_audio(track_obj, target_sr=None, mono=True):
    """
    Robust loader:
    1) try track_obj.audio
    2) if it fails and track_obj has .path, try torchaudio.load(path)
    3) if torchaudio fails, try ffmpeg decode
    4) if everything fails, raise an error
    """
    import subprocess, shutil

    if target_sr is None:
        target_sr = SR

    def _ffmpeg_decode_to_float32_mono(path, sr_out):
        ff = shutil.which("ffmpeg")
        if ff is None:
            raise RuntimeError("ffmpeg not found in PATH (needed as fallback).")
        cmd = [
            ff, "-nostdin", "-v", "error",
            "-i", str(path),
            "-ac", "1",
            "-ar", str(int(sr_out)),
            "-f", "f32le",
            "pipe:1",
        ]
        raw = subprocess.check_output(cmd, stderr=subprocess.PIPE)
        y = np.frombuffer(raw, dtype=np.float32)
        if y.size == 0:
            raise RuntimeError(f"ffmpeg decoded empty audio: {path}")
        return torch.from_numpy(y.copy()), int(sr_out)

    try:
        wav, sr = track_obj.audio
        wav = torch.tensor(wav).float()
        if wav.ndim > 1 and mono:
            wav = wav.mean(dim=0)
        if sr != target_sr:
            wav = torchaudio.functional.resample(wav, sr, target_sr)
            sr = target_sr
        return wav, sr

    except Exception as e1:
        path = getattr(track_obj, "path", None)

        if "BAD_AUDIO_COUNT" not in globals():
            globals()["BAD_AUDIO_COUNT"] = 0
        if globals()["BAD_AUDIO_COUNT"] < 5:
            print(f"[warn] audio load failed via track_obj.audio: {repr(e1)}")
            if path is not None:
                print(f"[warn] trying fallbacks for: {path}")
        globals()["BAD_AUDIO_COUNT"] += 1

        if path is not None:
            try:
                w, sr = torchaudio.load(str(path))
                if w.ndim == 2 and mono:
                    w = w.mean(dim=0)
                else:
                    w = w.squeeze(0)
                w = w.float()
                if sr != target_sr:
                    w = torchaudio.functional.resample(w, sr, target_sr)
                    sr = target_sr
                return w, sr
            except Exception:
                pass

            try:
                w, sr = _ffmpeg_decode_to_float32_mono(path, target_sr)
                if w.ndim > 1 and mono:
                    w = w.mean(dim=0)
                return w.float(), sr
            except Exception:
                pass

        raise RuntimeError(f"Unable to decode audio: {path}") from e1


In [ ]:
class MTATAudioDataset(Dataset):
    def __init__(self, tracks_dict, track_ids, device, sample_rate=None, audio_duration=None, mlb=None):
        self.tracks = tracks_dict
        self.track_ids = track_ids
        # Kept for the original constructor signature; samples stay on CPU.
        self.device = device  
        self.sample_rate = int(SR if sample_rate is None else sample_rate)
        self.audio_duration = float(CLIP_S if audio_duration is None else audio_duration)
        self.n_samples = int(self.sample_rate * self.audio_duration)

        tag_lists = []
        for tid in self.track_ids:
            tags = list(getattr(self.tracks[tid], "tags", []) or [])
            tag_lists.append(tags)

        if mlb is None:
            self.mlb = MultiLabelBinarizer()
            self.Y = self.mlb.fit_transform(tag_lists).astype(np.float32)
        else:
            self.mlb = mlb
            self.Y = self.mlb.transform(tag_lists).astype(np.float32)

    def __len__(self):
        return len(self.track_ids)

    def __getitem__(self, idx):
        tid = self.track_ids[idx]
        t = self.tracks[tid]

        wav, sr = load_track_audio(t, target_sr=self.sample_rate, mono=True)

        if wav.numel() < self.n_samples:
            wav = F.pad(wav, (0, self.n_samples - wav.numel()))
            start = 0
        else:
            start = random.Random(f"{CFG['seed']}:{tid}").randint(0, wav.numel() - self.n_samples)

        # Move samples to the selected device in the extraction loop.
        x = wav[start:start + self.n_samples].float().cpu()
        y = torch.tensor(self.Y[idx]).float().cpu()

        return {"audio": x, "labels": y, "tid": tid}

train_audio = MTATAudioDataset(tracks, train_ids, device=device, sample_rate=SR, audio_duration=CLIP_S, mlb=None)
val_audio   = MTATAudioDataset(tracks, val_ids, device=device, sample_rate=SR, audio_duration=CLIP_S, mlb=train_audio.mlb)
test_audio  = MTATAudioDataset(tracks, test_ids, device=device, sample_rate=SR, audio_duration=CLIP_S, mlb=train_audio.mlb)

n_classes = len(train_audio.mlb.classes_)
print("n_classes:", n_classes)
print("Example classes:", train_audio.mlb.classes_[:20])

resource_sample()


## Frozen embeddings and cache

In [ ]:
MODEL_ID = str(CFG.get("repr_model_id", "mtg-upf/clap_omarrq_mp_small_music"))
LAYERS   = list(CFG.get("emb_layers", [6]))
BS_EMB   = int(CFG.get("batch_size", 32))

def _safe_tag(s: str) -> str:
    return "".join([c if (c.isalnum() or c in ["-", "_"]) else "_" for c in str(s)])

def get_cache_paths(split_name: str, model_id: str, layers: list, sr: int, clip_s: float):
    layers_tag = "_".join(map(str, layers))
    mid = _safe_tag(model_id)
    emb_path = cache_dir / f"emb_{split_name}_{mid}_sr{sr}_cs{clip_s}_layers{layers_tag}.pt"
    y_path   = cache_dir / f"y_{split_name}_{mid}_sr{sr}_cs{clip_s}_layers{layers_tag}.pt"
    tid_path = cache_dir / f"tid_{split_name}_{mid}_sr{sr}_cs{clip_s}_layers{layers_tag}.json"
    return emb_path, y_path, tid_path

def _last_tensor(x):
    # omar_rq a veces devuelve lista/tupla por layers
    if isinstance(x, (list, tuple)):
        return x[-1]
    return x

def _as_BD(emb, B: int) -> torch.Tensor:
    """
    Convert OMAR-RQ embeddings to (B, D).

    Handles common shapes:
    * (B, D)
    * (B, T, D) or (T, B, D)
    * (1, B, T, D) or (L, B, T, D) : my case
    * any ndim>=3 where one axis == B
    """
    if isinstance(emb, (list, tuple)):
        emb = emb[-1]
    if not isinstance(emb, torch.Tensor):
        raise TypeError(f"Expected torch.Tensor, got {type(emb)}")

    # 2D: already (B,D) or transposed
    if emb.ndim == 2:
        if emb.shape[0] == B:
            return emb
        if emb.shape[1] == B:
            return emb.t()
        raise RuntimeError(f"2D emb but no dim matches batch B={B}: shape={tuple(emb.shape)}")

    # ndim >= 3: locate batch axis, move it to front, then pool everything except last dim
    if emb.ndim >= 3:
        if B not in emb.shape:
            raise RuntimeError(f"{emb.ndim}D emb but no dim matches batch B={B}: shape={tuple(emb.shape)}")

        # pick first axis that matches B
        b_axis = [i for i, s in enumerate(emb.shape) if s == B][0]
        if b_axis != 0:
            perm = [b_axis] + [i for i in range(emb.ndim) if i != b_axis]
            emb = emb.permute(*perm)  # (B, ...)

        # now emb is (B, d1, d2, ..., D?) and we want (B, D)
        # assumes embedding dim is last axis... pool over middle axes
        if emb.ndim == 3:
            # (B, T, D) or (B, D, T) isn't expected after permute but keep robust
            # decide which axis is D by taking the largest of the last two dims
            if emb.shape[2] >= emb.shape[1]:
                return emb.mean(dim=1)  # (B, D)
            else:
                return emb.mean(dim=2)  # (B, D)

        # ndim >= 4 (e.g., (B, 1, 25, 512)): mean over axes 1..ndim-2 => (B, 512)
        reduce_dims = list(range(1, emb.ndim - 1))
        if len(reduce_dims) > 0:
            emb = emb.mean(dim=reduce_dims)

        if emb.ndim == 1:
            emb = emb.unsqueeze(1)
        if emb.ndim != 2:
            emb = emb.view(emb.shape[0], -1)

        if emb.shape[0] != B:
            raise RuntimeError(f"After pooling, batch mismatch: got {tuple(emb.shape)}, expected B={B}")
        return emb

    raise RuntimeError(f"Unsupported emb shape for B={B}: shape={tuple(emb.shape)}")

@torch.no_grad()
def compute_or_load_embeddings(
    split_name,
    audio_dataset,
    model_id=None,
    layers=None,
    batch_size=None,
):
    model_id = MODEL_ID if model_id is None else str(model_id)
    layers = LAYERS if layers is None else list(layers)
    batch_size = BS_EMB if batch_size is None else int(batch_size)

    sr = int(CFG.get("sample_rate", 16000))
    clip_s = float(CFG.get("clip_seconds", 2.0))

    identity = cache_digest({
        "version": 1, "model": model_id, "layers": layers, "sr": sr, "seconds": clip_s,
        "seed": CFG["seed"], "data_revision": CFG["data_revision"],
        "track_ids": audio_dataset.track_ids, "classes": audio_dataset.mlb.classes_.tolist(),
        "labels": audio_dataset.Y.tolist(), "crop": "per-track-seeded",
    })
    emb_path, y_path, tid_path = get_cache_paths(f"{split_name}_{identity}", model_id, layers, sr, clip_s)
    expected_n = len(audio_dataset)

    if emb_path.exists() and y_path.exists() and tid_path.exists():
        print(color.BOLD + f"[cache] Loading {split_name} embeddings from disk..." + color.END)
        X = torch.load(emb_path, map_location="cpu")
        Y = torch.load(y_path, map_location="cpu")
        tids = json.loads(tid_path.read_text(encoding="utf-8"))

        ok = (
            isinstance(X, torch.Tensor) and isinstance(Y, torch.Tensor)
            and X.ndim == 2 and Y.ndim == 2
            and X.shape[0] == expected_n and Y.shape[0] == expected_n
            and tids == list(audio_dataset.track_ids)
            and Y.shape[1] == audio_dataset.Y.shape[1]
            and torch.equal(Y, torch.from_numpy(audio_dataset.Y))
            and bool(torch.isfinite(X).all())
        )
        if ok:
            return X, Y, tids

        print(color.YELLOW + f"[warn] cache mismatch for {split_name} -> recompute and overwrite" + color.END)
        for p in [emb_path, y_path, tid_path]:
            try:
                p.unlink()
            except Exception:
                pass

    print(color.BOLD + f"[compute] Extracting {split_name} embeddings..." + color.END)
    rep = get_model(model_id=model_id, device=device)
    rep.eval()

    loader = DataLoader(
        audio_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=int(CFG.get("num_workers", 0)),
        pin_memory=bool(CFG.get("pin_memory", False)),
    )

    X_list, Y_list, T_list = [], [], []
    target_D = None

    if "EMB_WARN_COUNT" not in globals():
        globals()["EMB_WARN_COUNT"] = 0

    for batch in tqdm(loader, desc=f"Embeddings {split_name}"):
        audio = batch["audio"]
        y = batch["labels"].detach().cpu()
        tids = list(batch["tid"])

        # =========== IMPORTANT: OMAR-RQ expects (B, T), no channel dim
        if audio.ndim == 3 and audio.shape[1] == 1:
            audio = audio.squeeze(1)  # (B,1,T) -> (B,T)
        elif audio.ndim > 2:
            audio = audio.reshape(audio.shape[0], -1)

        audio = audio.to(device, non_blocking=True).float()
        B = int(audio.shape[0])

        emb_raw = rep.extract_embeddings(audio, layers=layers)
        emb_t = _last_tensor(emb_raw)

        # CASE A: model collapses the batch and returns (1, D) even when B>1
        if isinstance(emb_t, torch.Tensor) and emb_t.ndim == 2 and emb_t.shape[0] == 1 and B > 1:
            if globals()["EMB_WARN_COUNT"] < 5:
                print(color.YELLOW + f"[warn] pooled batch embedding {tuple(emb_t.shape)} for B={B} -> per-sample fallback" + color.END)
            globals()["EMB_WARN_COUNT"] += 1

            rows = []
            for i in range(B):
                e_i = rep.extract_embeddings(audio[i:i+1], layers=layers)
                rows.append(_as_BD(e_i, 1).detach().cpu())
            emb = torch.cat(rows, dim=0)

        else:
            try:
                emb = _as_BD(emb_raw, B).detach().cpu()
            except Exception as e:
                if globals()["EMB_WARN_COUNT"] < 5:
                    print(color.YELLOW + f"[warn] batch emb shape issue -> per-sample fallback | {repr(e)}" + color.END)
                    if isinstance(emb_t, torch.Tensor):
                        print(color.YELLOW + f"       emb_last shape: {tuple(emb_t.shape)} | B={B}" + color.END)
                globals()["EMB_WARN_COUNT"] += 1

                rows = []
                for i in range(B):
                    e_i = rep.extract_embeddings(audio[i:i+1], layers=layers)
                    rows.append(_as_BD(e_i, 1).detach().cpu())
                emb = torch.cat(rows, dim=0)

        if target_D is None:
            target_D = int(emb.shape[1])
        elif int(emb.shape[1]) != target_D:
            raise ValueError("Embedding dimension changed within a split; inspect encoder output shapes.")
        if not torch.isfinite(emb).all():
            raise ValueError("Non-finite audio embeddings.")

        X_list.append(emb)
        Y_list.append(y)
        T_list.extend(tids)

        if random.random() < 0.05:
            resource_sample()

    X = torch.cat(X_list, dim=0)
    Y = torch.cat(Y_list, dim=0)

    if not (X.shape[0] == Y.shape[0] == len(T_list) == expected_n):
        raise RuntimeError(
            f"Alignment error in {split_name}: "
            f"X0={X.shape[0]} Y0={Y.shape[0]} tids={len(T_list)} expected={expected_n}"
        )

    torch.save(X, emb_path)
    torch.save(Y, y_path)
    tid_path.write_text(json.dumps(T_list, ensure_ascii=False, indent=2), encoding="utf-8")

    print(color.BOLD + f"[saved] {split_name}: X={tuple(X.shape)} Y={tuple(Y.shape)}" + color.END)
    return X, Y, T_list


# run
X_train, Y_train, tids_train = compute_or_load_embeddings("train", train_audio)
X_val,   Y_val,   tids_val   = compute_or_load_embeddings("val",   val_audio)
X_test,  Y_test,  tids_test  = compute_or_load_embeddings("test",  test_audio)

print(color.BOLD + "Embeddings shapes" + color.END)
print("Train:", X_train.shape, Y_train.shape)
print("Val:  ", X_val.shape, Y_val.shape)
print("Test: ", X_test.shape, Y_test.shape)

resource_sample()


## MLP classifier

In [ ]:
class EmbeddingDataset(Dataset):
    def __init__(self, X, Y):
        self.X = X.float()
        self.Y = Y.float()

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return {"embeddings": self.X[idx], "labels": self.Y[idx]}

train_ds = EmbeddingDataset(X_train, Y_train)
val_ds   = EmbeddingDataset(X_val,   Y_val)
test_ds  = EmbeddingDataset(X_test,  Y_test)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)

class MLP(nn.Module):
    def __init__(self, input_size, hidden=256, output_size=50, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, output_size),
        )

    def forward(self, x):
        return self.net(x)

input_dim = X_train.shape[1]
model = MLP(input_size=input_dim, hidden=CFG["mlp_hidden"], output_size=n_classes, dropout=CFG["mlp_dropout"]).to(device)

loss_fn = nn.BCEWithLogitsLoss()
opt = torch.optim.Adam(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])

print(model)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

resource_sample()


In [ ]:
# checkpoints: save best model + resume support
def save_checkpoint(path, model, optimizer, epoch, history, extra=None):
    payload = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optim_state": optimizer.state_dict(),
        "history": history,
        "extra": extra or {},
    }
    torch.save(payload, path)

def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["model_state"])
    if optimizer is not None and "optim_state" in ckpt:
        optimizer.load_state_dict(ckpt["optim_state"])
    return ckpt

best_model_path = run_dir / "best_model.pt"
last_ckpt_path  = run_dir / "last_checkpoint.pt"


## Training

Select the checkpoint by validation micro average precision, evaluated every three epochs. The original notebook called `average_precision_score` “PR-AUC”; it is reported here as average precision (AP). Early stopping counts validation checks, not individual epochs.

In [ ]:
from tqdm.auto import tqdm  

@torch.no_grad()
def predict_probs(model, loader, desc="Predict", show_pbar=True):
    model.eval()
    ys, ps = [], []
    it = tqdm(loader, desc=desc, leave=False) if show_pbar else loader
    for batch in it:
        x = batch["embeddings"].to(device)
        y = batch["labels"].cpu().numpy()
        logits = model(x)
        p = torch.sigmoid(logits).detach().cpu().numpy()
        ys.append(y)
        ps.append(p)
    return np.vstack(ys), np.vstack(ps)

def train_one_epoch(model, loader, desc="Train", show_pbar=True):
    model.train()
    total = 0.0
    it = tqdm(loader, desc=desc, leave=False) if show_pbar else loader
    for batch in it:
        x = batch["embeddings"].to(device)
        y = batch["labels"].to(device)

        logits = model(x)
        loss = loss_fn(logits, y)

        opt.zero_grad()
        loss.backward()
        opt.step()

        total += float(loss.item())

        # mostrar loss en la barra
        if show_pbar:
            it.set_postfix(loss=f"{loss.item():.4f}")

    return total / max(1, len(loader))

@torch.no_grad()
def eval_loss(model, loader, desc="Val loss", show_pbar=True):
    model.eval()
    total = 0.0
    it = tqdm(loader, desc=desc, leave=False) if show_pbar else loader
    for batch in it:
        x = batch["embeddings"].to(device)
        y = batch["labels"].to(device)
        logits = model(x)
        loss = loss_fn(logits, y)
        total += float(loss.item())
        if show_pbar:
            it.set_postfix(loss=f"{loss.item():.4f}")
    return total / max(1, len(loader))

def safe_auc_pr(y_true, y_prob):
    out = {}
    try:
        out["roc_auc_macro"] = float(roc_auc_score(y_true, y_prob, average="macro"))
    except Exception:
        out["roc_auc_macro"] = None
    try:
        out["roc_auc_micro"] = float(roc_auc_score(y_true, y_prob, average="micro"))
    except Exception:
        out["roc_auc_micro"] = None
    try:
        out["pr_auc_macro"] = float(average_precision_score(y_true, y_prob, average="macro"))
    except Exception:
        out["pr_auc_macro"] = None
    try:
        out["pr_auc_micro"] = float(average_precision_score(y_true, y_prob, average="micro"))
    except Exception:
        out["pr_auc_micro"] = None
    return out

history = {"epoch": [], "train_loss": [], "val_loss": [], "val_roc_auc_macro": [], "val_roc_auc_micro": [], "val_pr_auc_macro": [], "val_pr_auc_micro": []}

best_val_pr_micro = -1.0
start_epoch = 1

# =================================== CFG-driven training knobs
n_epochs   = int(CFG.get("epochs", 10))
# cada cuántos epochs calcular AUC/PR
eval_every = int(CFG.get("eval_every", 3))              
# 0 => sin early stop
patience   = int(CFG.get("early_stop_patience", 0))     
# NUEVO: controlar barras
show_pbar  = bool(CFG.get("show_pbar", True))           
no_improve = 0

# Si start_epoch ya pasó el total de epochs (por CFG más pequeño) NO ENTRENARÁ MAS
if start_epoch > n_epochs:
    print(f"[info] start_epoch={start_epoch} > n_epochs={n_epochs} -> skipping training loop.")
else:
    for epoch in range(start_epoch, n_epochs + 1):
        tr = train_one_epoch(model, train_loader, desc=f"Train epoch {epoch:02d}", show_pbar=show_pbar)
        va = eval_loss(model, val_loader, desc=f"Val loss epoch {epoch:02d}", show_pbar=show_pbar)

        metrics = {"roc_auc_macro": None, "roc_auc_micro": None, "pr_auc_macro": None, "pr_auc_micro": None}
        do_eval = (epoch % eval_every == 0) or (epoch == n_epochs)
        if do_eval:
            Yt, Yp = predict_probs(model, val_loader, desc=f"Val predict epoch {epoch:02d}", show_pbar=show_pbar)
            metrics = safe_auc_pr(Yt, Yp)

        history["epoch"].append(epoch)
        history["train_loss"].append(tr)
        history["val_loss"].append(va)
        history["val_roc_auc_macro"].append(metrics["roc_auc_macro"])
        history["val_roc_auc_micro"].append(metrics["roc_auc_micro"])
        history["val_pr_auc_macro"].append(metrics["pr_auc_macro"])
        history["val_pr_auc_micro"].append(metrics["pr_auc_micro"])

        val_pr_micro = metrics["pr_auc_micro"]
        print(f"epoch {epoch:02d} | train_loss={tr:.4f} | val_loss={va:.4f} | val_pr_micro={val_pr_micro}")

        # Save best (based on val PR-AUC micro)
        improved = False
        if val_pr_micro is not None and val_pr_micro > best_val_pr_micro:
            best_val_pr_micro = val_pr_micro
            improved = True
            print(f"[best] New best val PR-AUC micro: {best_val_pr_micro:.4f} -> saving best_model.pt")
            torch.save(model.state_dict(), best_model_path)

        # Early stopping (solo cuenta cuando evaluamos PR-AUC)
        if patience > 0 and do_eval:
            if improved:
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"[early-stop] no improvement in val PR-AUC micro for {patience} eval checks. Stopping.")
                    save_checkpoint(
                        last_ckpt_path,
                        model=model,
                        optimizer=opt,
                        epoch=epoch,
                        history=history,
                        extra={"best_val_pr_micro": best_val_pr_micro, "no_improve": no_improve},
                    )
                    resource_sample()
                    break

        # Save last checkpoint (every epoch)
        save_checkpoint(
            last_ckpt_path,
            model=model,
            optimizer=opt,
            epoch=epoch,
            history=history,
            extra={"best_val_pr_micro": best_val_pr_micro, "no_improve": no_improve},
        )

        resource_sample()


## Test evaluation

Reload the best validation checkpoint before computing and saving test metrics. The original notebook first evaluated the last training state and later loaded the best checkpoint, which produced two different test values. See the results document for both historical outputs.

In [ ]:
if not best_model_path.exists():
    raise RuntimeError("No best checkpoint was saved; inspect the validation metrics.")
model.load_state_dict(torch.load(best_model_path, map_location=device))
Yt_test, Yp_test = predict_probs(model, test_loader)
test_metrics = safe_auc_pr(Yt_test, Yp_test)

print("TEST METRICS")
for k, v in test_metrics.items():
    print(f"{k}: {v}")

results = {
    "device": str(device),
    "platform": platform.platform(),
    "torch_version": torch.__version__,
    "repr_model_id": str(MODEL_ID),
    "emb_layers": list(LAYERS),
    "input_dim": int(input_dim),
    "n_classes": int(n_classes),
    "epochs_configured": int(n_epochs),
    "epochs_ran": len(history["epoch"]),
    "checkpoint": "best_validation_micro_ap",
    "config": CFG,
    "final_test_metrics": test_metrics,
    "history": history,
}

results_path = run_dir / "results_lab9_mtat_transfer.json"
results_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", results_path)

resource_sample()


## Prior and random baselines

In [ ]:
prior = Y_train.numpy().mean(axis=0, keepdims=True)
prior_probs = np.repeat(prior, Yt_test.shape[0], axis=0)
random_probs = np.random.default_rng(0).random(Yt_test.shape)
baseline_results = {
    "prior": safe_auc_pr(Yt_test, prior_probs),
    "random": safe_auc_pr(Yt_test, random_probs),
    "mlp": test_metrics,
}
(run_dir / "baselines.json").write_text(json.dumps(baseline_results, indent=2))
baseline_results


## Training curves

In [ ]:
epochs = history["epoch"]

plt.figure(figsize=(8,4))
plt.plot(epochs, history["train_loss"], label="train_loss")
plt.plot(epochs, history["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss curves")
plt.legend()
plt.tight_layout()
plt.show()

def plot_metric(metric_name, title):
    vals = history[metric_name]
    xs = [e for e, v in zip(epochs, vals) if v is not None]
    ys = [v for v in vals if v is not None]
    if len(xs) == 0:
        print(f"No values for {metric_name}")
        return
    plt.figure(figsize=(8,4))
    plt.plot(xs, ys, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel(metric_name)
    plt.title(title)
    plt.tight_layout()
    plt.show()

plot_metric("val_roc_auc_micro", "Validation ROC-AUC (micro)")
plot_metric("val_pr_auc_micro", "Validation PR-AUC (micro)")

resource_sample()


## Resource summary

In [ ]:
print("Stopping emissions tracker...")
emissions = emissions_tracker.stop() if emissions_tracker is not None else None

# Summaries
peak_ram = max(resource_log.ram_used_gb) if resource_log.ram_used_gb else None
peak_cpu = max(resource_log.cpu_percent) if resource_log.cpu_percent else None
peak_cuda_alloc = max(resource_log.cuda_mem_alloc_gb) if resource_log.cuda_mem_alloc_gb else None

summary = {
    "emissions_kg_co2eq": emissions,
    "peak_ram_gb": peak_ram,
    "peak_cpu_percent": peak_cpu,
    "peak_cuda_mem_alloc_gb": peak_cuda_alloc,
}

summary_path = run_dir / "resource_emissions_summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", summary_path)
print(summary)

# Plots
ts = resource_log.timestamps_s

plt.figure(figsize=(10,4))
plt.plot(ts, resource_log.cpu_percent)
plt.xlabel("Time (s)")
plt.ylabel("CPU %")
plt.title("CPU usage over time")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10,4))
plt.plot(ts, resource_log.ram_used_gb)
plt.xlabel("Time (s)")
plt.ylabel("RAM used (GB)")
plt.title("RAM usage over time")
plt.tight_layout()
plt.show()

if torch.cuda.is_available():
    plt.figure(figsize=(10,4))
    plt.plot(ts, resource_log.cuda_mem_alloc_gb, label="allocated")
    plt.plot(ts, resource_log.cuda_mem_reserved_gb, label="reserved")
    plt.xlabel("Time (s)")
    plt.ylabel("CUDA memory (GB)")
    plt.title("CUDA memory over time")
    plt.legend()
    plt.tight_layout()
    plt.show()


## What this experiment establishes

The saved original run outperformed the prior and random baselines under its clip-level protocol. It does not establish performance on unseen artists, an official MTAT benchmark, or a new run of this edited notebook. Historical caches and model revisions were not fully recorded. See [results and limitations](../docs/results.md).